In [ ]:
# Environment pre-configured via .binder/ for Binder (GESIS / MyBinder).
# For local use: activate and instantiate the environment in this directory:
#   julia --project=. -e 'using Pkg; Pkg.instantiate()'
# then launch Jupyter with: julia --project=. -e 'using IJulia; notebook()'

In [ ]:
using CSV, DataFrames; ENV["COLUMNS"]=160 # for displaying more columns of a dataframe
using StaticArrays
using BenchmarkTools
using Plots, LaTeXStrings; using Plots.PlotMeasures; 
default(framestyle = :box, minorticks = 5, fg_color_legend = :lightgray);
using QuadGK; using Interpolations
using JuMinuit

In [ ]:
const unit = 1.0;
const fpi = 92.21unit; const mpic = 139.57018unit; const mpi0 = 134.9766unit;
const meta = 547.862unit; const mkc = 493.677unit; const mk0 = 497.614unit;
const mpi = (2mpic + mpi0)/3; const mk = (mkc + mk0)/2
const μ = 770.0unit;
const ϵ = eps();

# best fit parameters from Gomez Nicola, Pelaez
const lecr0 = [0.56e-3, 1.21e-3, -2.79e-3, -0.36e-3, 1.4e-3, 0.07e-3, -0.44e-3, 0.78e-3];

# NLO CHPT LECs
const lec_chpt = (0.4e-3, 1.35e-3, -3.5e-3, -0.3e-3, 1.4e-3, -0.2e-3, -0.4e-3, 0.9e-3)

const paras0 = [lecr0..., 1e-4];

# Data

The data for the 00 wave up to 970 MeV are taken from the table in the appendix of the GKPRY paper:

*The Pion-pion scattering amplitude. IV: Improved analysis with once subtracted Roy-like equations up to 1100 MeV*
 R. Garcia-Martin, R. Kaminski, J.R. Pelaez, J. Ruiz de Elvira, F.J. Yndurain, [Phys.Rev.D 83 (2011) 074004](https://doi.org/10.1103/PhysRevD.83.074004) [[1102.2183 [hep-ph]](https://arxiv.org/abs/1102.2183)]
The data points on $\delta_0^0$ above 970 MeV and $\eta_0^0$ taken read off from Figs. 15 and 17, respectively:

<img src="./datajl/pipi00_Roy-GKPY_PRD83_074004.png" style="height: 250px;"/>  <img src="./datajl/eta00_Roy-GKPY_PRD83_074004.png" style="height: 250px;"/> 

I tried to take data for the 11 and 20 waves above 970 MeV from:
*The Pion-pion scattering amplitude. III. Improving the analysis with forward dispersion relations and Roy equations*
R. Kaminski, J.R. Pelaez, F.J. Yndurain, [Phys.Rev.D 77 (2008) 054015](https://doi.org/10.1103/PhysRevD.77.054015) [[0710.1150 [hep-ph]](https://arxiv.org/abs/0710.1150)]

<img src="./datajl/pipi11_Roy-KPY_PRD77_054015.png" style="height: 250px;"/>  <img src="./datajl/pipi20_Roy-KPY_PRD77_054015.png" style="height: 250px;"/> 

However, the $\delta_0^2$ values do not match: the central value at 970 MeV in the figure is apperantly larger than that givein in the table. $\delta_1^1$ is roughly okay.

In [ ]:
# data from the GKPRY paper
data_GKPRY_ππ00_df = DataFrame(CSV.File("./datajl/pipi/pipi00_Roy-GKPY_PRD83_074004.dat", header = [:w, :δ, :err], 
        delim=' ', ignorerepeated=true))
data_GKPRY_ππ11_df = DataFrame(CSV.File("./datajl/pipi/pipi11_Roy-GKPY_PRD83_074004.dat", header = [:w, :δ, :err], 
        delim=' ', ignorerepeated=true))
data_GKPRY_ππ20_df = DataFrame(CSV.File("./datajl/pipi/pipi20_Roy-GKPY_PRD83_074004.dat", header = [:w, :δ, :err], 
        delim=' ', ignorerepeated=true))
data_GKPRY_ππ00_eta_df = DataFrame(CSV.File("./datajl/pipikk/eta00_Roy-GKPY_PRD83_074004.dat", header = [:w, :η, :err], 
        delim=' ', ignorerepeated=true));
# join(data_GKPRY_ππ00_df, data_GKPRY_ππ11_df, data_GKPRY_ππ20_df, on = :w, makeunique=true)

In [ ]:
# Cohen et al., PRD22,2595 (1980)
data_ππkk00_Cohen_df = DataFrame(CSV.File("./datajl/pipikk/pipikk-Cohen-PRD22_2595.dat", header = [:w, :δ, :err], 
        delim='\t', ignorerepeated=true))
# Martin and Ozmutlu, NPB158,520 (1979); 9 points below 1.2 GeV
data_ππkk00_Martin_df = DataFrame(CSV.File("./datajl/pipikk/pipikk-Martin-NPB158_520.dat", header = [:w, :δ, :err],
        delim='\t', ignorerepeated=true));

# recommended inelasticity data by Jose. 
# from Kaminski, Leniak, Rybicki, Z.Phys.C 74 (1997) 79-91, https://inspirehep.net/literature/419770
# 30 points below 1.2 GeV
data_ππ00_eta_Kaminski_df = DataFrame(CSV.File("./datajl/pipikk/S0ine-Kaminski-ZPC74_79.dat", 
        header = [:w, :η, :err], delim='\t', ignorerepeated=true));

In [ ]:
# Armstrong et al. [WA76], Z.Phys.C 52 (1991) 389-396
data_πη_WA76_df = DataFrame(CSV.File("./datajl/pieta_ZPC52_389.csv", header = [:w, :y, :err]));
# interpolate the background extracted from data
data_πη_WA76_bg_df = sort(DataFrame(CSV.File("./datajl/pieta_bg_ZPC52_389.csv", header = [:w, :y])), :w)
const data_πη_WA76_bg = LinearInterpolation(data_πη_WA76_bg_df.w, data_πη_WA76_bg_df.y);

In [ ]:
# S-wave Kπ scattering phase shifts
# Aston et al, NPB296, 493; 21 points below 1.2 GeV
data_πK10_Aston_df = DataFrame(CSV.File("./datajl/kpi/S-Aston-PhaseShift-New.dat", 
        header = [:w, :δ, :err], delim="\t", ignorerepeated=true))
# Estabrooks et al., NPB133, 490; 19 points below 1.2 GeV
data_πK10_Estabrooks_df = DataFrame(CSV.File("./datajl/kpi/S-Estabrooks-PhaseShift-New.dat", 
        header = [:w, :δ, :err], delim='\t', ignorerepeated=true));
# Mercer et al., NPB32, 381; 14 points below 1.2 GeV
data_πK10_Mercer_df = DataFrame(CSV.File("./datajl/kpi/S-Mercer-PhaseShift-NPB32_381.dat", 
        header = [:w, :δ, :err]))
# Bingham et al., NPB296, 493; 19 points below 1.2 GeV
data_πK10_Bingham_df = DataFrame(CSV.File("./datajl/kpi/S-Bingham-PhaseShift-NPB296_493.dat", 
        header = [:w, :δ, :err]))
# Baker et al., NPB99, 211
data_πK10_Baker_df = DataFrame(CSV.File("./datajl/kpi/S-Baker-PhaseShift-NPB99_211.dat", 
        header = [:w, :δ, :err], delim=" ", ignorerepeated=true))
# BaBar, PRD83, 072001
data_πK10_BaBar_df = DataFrame(CSV.File("./datajl/kpi/S-BaBar-PhaseShift_PRD83_072001.dat", 
        header = [:w, :δ, :err], delim="\t", ignorerepeated=true))
# BES, PRD94, 032001
data_πK10_BES_df = DataFrame(CSV.File("./datajl/kpi/S-BES-PhaseShift_PRD94_032001.dat", 
        header = [:w, :δ, :err], delim=" ", ignorerepeated=true))

# 21 points below 1.2 GeV
data_πK11_Aston_df = DataFrame(CSV.File("./datajl/kpi/P-Aston-NPB296_493.dat", 
        header = [:w, :δ, :err], delim=' ', ignorerepeated=true))
# 23 points up to 1.2 GeV
data_πK11_Estabrooks_df = DataFrame(CSV.File("./datajl/kpi/P-Estabrooks-NPB133_490.dat", 
        header = [:w, :δ, :err], delim=' ', ignorerepeated=true))
# 14 points below 1.2 GeV
data_πK11_Mercer_df = DataFrame(CSV.File("./datajl/kpi/P-Mercer_PhaseShift_NPB32_381.dat", 
        header = [:w, :δ, :err], delim='\t', ignorerepeated=true))

# 10 data points below 1.2 GeV
data_πK30_Estabrooks_df = DataFrame(CSV.File("./datajl/kpi/S32-Estabrooks-NPB133_490.dat", 
        header = [:w, :δ, :err], delim=' ', ignorerepeated=true))
data_πK30_Lingin_df = DataFrame(CSV.File("./datajl/kpi/S32-Linglin_NPB57_64.dat", 
        header = [:w, :δ, :err], delim='\t', ignorerepeated=true))
# 14 points below 1.2 GeV
data_πK30_Mercer_df = DataFrame(CSV.File("./datajl/kpi/S32-Mercer-PhaseShift-NPB32_381.dat", 
        header = [:w, :δ, :err], delim='\t', ignorerepeated=true));

In [ ]:
data_πK10_combined_df = sort(vcat(data_πK10_Aston_df, data_πK10_Estabrooks_df, data_πK10_Mercer_df, data_πK10_Bingham_df, data_πK10_Baker_df,
        data_πK10_BaBar_df, data_πK10_BES_df), :w);
data_πK11_combined_df = sort(vcat(data_πK11_Aston_df, data_πK11_Estabrooks_df, data_πK11_Mercer_df), :w);
data_πK30_combined_df = sort(vcat(data_πK30_Estabrooks_df, data_πK30_Lingin_df, data_πK30_Mercer_df), :w);

data_ππKK_combined_df = sort(vcat(data_ππkk00_Cohen_df, data_ππkk00_Martin_df), :w);

In [ ]:
const data_ππ00 = Data(data_GKPRY_ππ00_df)
const data_ππ11 = Data(data_GKPRY_ππ11_df)
const data_ππ20 = Data(data_GKPRY_ππ20_df)
const data_ππ00_eta = Data(data_ππ00_eta_Kaminski_df[1:30,:]);
const data_ππKK00 = Data(data_ππKK_combined_df[ data_ππKK_combined_df.:w .<=1200, :]);
const data_πK10 = Data(data_πK10_combined_df[ data_πK10_combined_df.:w .<=1200, :])
const data_πK11 = Data(data_πK11_combined_df[ data_πK11_combined_df.:w .<=1200, :])
const data_πK30 = Data(data_πK30_combined_df[ data_πK30_combined_df.:w .<=1200, :])
const data_πη = Data(data_πη_WA76_df);

In [ ]:
pyplot()
let p1, p2, p3, p4
    p1 = @plt_data data_ππ00 label = L"\delta_{0}^0" marker = (:circle, :dodgerblue)
    @plt_data! data_ππ11 label = L"\delta_{1}^1" marker = (:circle, :red2)
    @plt_data! data_ππ20 label = L"\delta_{2}^0" marker = (:circle, :green3)
    
    p2 = @plt_data Data(data_πK10_Estabrooks_df)[1:19] label = L"\delta_{1/2}^0" marker = (:rect, :dodgerblue)
    @plt_data! Data(data_πK10_Aston_df)[1:21] label = "" marker = (:diamond, :dodgerblue)
    @plt_data! Data(data_πK10_Mercer_df)[1:14] label = "" marker = (:utriangle, :dodgerblue)
    @plt_data! Data(data_πK10_Bingham_df)[1:19] label = "" marker = (:dtriangle, :dodgerblue)
    @plt_data! Data(data_πK10_Baker_df) label = "" marker = (:pentagon, :dodgerblue)
    @plt_data! Data(data_πK10_BES_df) label = "" marker = (:rtriangle, :dodgerblue)
    @plt_data! Data(data_πK10_BaBar_df) label = "" marker = (:ltriangle, :dodgerblue)
    
    @plt_data! Data(data_πK11_Estabrooks_df)[1:23] label = L"\delta_{1/2}^1" marker = (:rect, :red2)
    @plt_data! Data(data_πK11_Aston_df)[1:21] label = "" marker = (:diamond, :red2)
    @plt_data! Data(data_πK11_Mercer_df)[1:14] label = "" marker = (:utriangle, :red2)
 
    @plt_data! Data(data_πK30_Estabrooks_df)[1:10] label = L"\delta_{3/2}^0" marker = (:rect, :green3)
    @plt_data! Data(data_πK30_Mercer_df)[1:14] label="" marker = (:utriangle, :green3)
    @plt_data! Data(data_πK30_Lingin_df) label="" marker = (:+, :darkgreen ) #, stroke(:darkgreen))

    p3 = @plt_data data_ππ00_eta label = L"\eta_{0}^0"
    p4 = @plt_data Data(data_ππkk00_Cohen_df) label = L"\delta_{0}^0 [\pi\pi\to K \bar K]"
    @plt_data! Data(data_ππkk00_Martin_df)[1:9] label = "" marker = :rect
    
    p5 = @plt_data data_πη label = L"d\sigma_{\pi\eta}/d\sqrt{s}"
    
    plot(p1, p2, p3, p4, p5, size=(1200,550))
end

# UCHPT

In [ ]:
struct TwoBodyChannel{T<:AbstractFloat} 
    m1::T
    m2::T
end

qon(s, m1, m2) = sqrt((s - (m1+m2)^2) * (s - (m1-m2)^2))/(2sqrt(s))

In [ ]:
const ππ = TwoBodyChannel(mpi, mpi)
const KK = TwoBodyChannel(mk, mk)
const ηη = TwoBodyChannel(meta, meta)
const πη = TwoBodyChannel(mpi, meta)
const Kπ = TwoBodyChannel(mk, mpi)
const Kη = TwoBodyChannel(mk, meta);

In [ ]:
# load some const definitions used in tmatrix.jl
include("src/init_const.jl");

In [ ]:
# amplitudes from LO and NLO CHPT
include("src/amplitudes.jl")
# amplitudes from IAM
include("src/tmatrix.jl")
# modifications to satisfy unitarity
include("src/unitarity_modification.jl")
include("src/phaseshifts.jl");

In [ ]:
const cmatrix22 = zeros(ComplexF64, 2, 2);
const cmatrix33 = zeros(ComplexF64, 3, 3);

In [ ]:
# checked with FORTRAN; 0 allocations with julia >=1.5.0
@time St4_11!(cmatrix22, 1040, lecr0) 

In [ ]:
# checked with FORTRAN; 0 allocations with julia >=1.5.0
@btime St4_00!(cmatrix33, 540, lecr0) samples=100

In [ ]:
# IJ = 11, ππ, KK; checked with FORTRAN
@btime Stiam_11(540, lecr0) samples=100

In [ ]:
# IJ = 00, ππ, KK, ηη; checked with FORTRAN
@time Stiam_00(540, lecr0) 

In [ ]:
@time Stiam_10(540, lecr0) 

Linear combinations of LECs:

| NLO amp. | LEC combinations from the NLO scattering amplitudes            |
|:---|:---|
|V4_I0_ππππ, V4_I1_ππππ  | $2L_1 + L_3$, $L_2$, $2L_4+L5$, $2L_6+L_8$ |
|V4_I0_ππkk, V4_I1_kπkπ, V4_I3_kπkπ | $L_1, L_2, L_3, L_4, L_5, 2L_6+L_8$ |
|V4_I0_kkkk, V4_I1_kkkk | $L_1, L_2, L_3, L_4, L_5, 2L_6+L_8$ |
|V4_I0_ππηη, V4_I1_πηπη  | $L_1, L_2, L_3, L_4, 2m_\eta^2 m_\pi^2(-L_5/3 + 2 L_6) + 4m_\pi^2 (- m_\eta^2 + m_\pi^2) L_7 + 2 m_\pi^4 L_8$ |
|V4_I0_kkηη | $L_1, L_2, L_3, L_4, L_5, 8m_\eta^2 m_K^2 (-L_4+L_6) + 2 (3 m_\eta^4 - 4 m_\eta^2 m_\pi^2 + m_\pi^4) L_7 + (6 m_\eta^4 - 3 m_\eta^2 m_\pi^2 + m_\pi^4) L_8$ |
|V4_I1_πηkk | $L_3, L_5, 6(m_\eta^4-m_\pi^4)(2L_7+ L_8)$|

In [ ]:
# πη invariant mass distribution
function σπη(w, paras)
    _m1, _m2 = πη.m1, πη.m2
    _c = @views paras[9]
    if w ≥ _m1 + _m2 + 0.0000001
        _q = qon(w^2, _m1, _m2)
        _tm = Stiam_10(w, @views paras[1:8])[2, 1]
        res = _c * _q * abs2(_tm ) + data_πη_WA76_bg(w) # _bg* _q^4 )
    else
        res = 0.0
    end
    return  res 
end

In [ ]:
function plt_compare(paras; jump = 900)
    _lec = Ref(paras[1:8])
    wv = 270:1.0:1200
    p1 = plot(wv, δ00.(wv, _lec; jump = jump), label="", xlim = (wv[1], wv[end]))
    plot!(wv, δ11.(wv, _lec), label="" )
    plot!(wv, δ20.(wv, _lec), label="" )
    @plt_data! data_ππ00 label = L"\delta_{0}^0" marker = (:circle, :dodgerblue)
    @plt_data! data_ππ11 label = L"\delta_{1}^1" marker = (:circle, :darkorange)
    @plt_data! data_ππ20 label = L"\delta_{2}^0" marker = (:circle, :green3) xlab=L"\sqrt{s}"*" [MeV]" ylab = L"\delta_{J}^I (\pi\pi\to \pi\pi)"
    hline!([0], color = :black, label=:none)    
    
    xv = 630:1.0:1200
    p2 = plot(xv, δhalf0.(xv, _lec), label="", xlim = (xv[1], xv[end]))
    plot!(xv, δhalf1.(xv, _lec), label="" )
    plot!(xv, δ3half0.(xv, _lec), label="" )
    @plt_data! Data(data_πK10_Estabrooks_df)[1:19] label = L"\delta_{1/2}^0" marker = (:rect, :dodgerblue)
    @plt_data! Data(data_πK10_Aston_df)[1:21] label = "" marker = (:diamond, :dodgerblue)
    @plt_data! Data(data_πK10_Mercer_df)[1:14] label = "" marker = (:utriangle, :dodgerblue)
    @plt_data! Data(data_πK10_Bingham_df)[1:19] label = "" marker = (:dtriangle, :dodgerblue)
    @plt_data! Data(data_πK10_Baker_df) label = "" marker = (:pentagon, :dodgerblue)
    @plt_data! Data(data_πK10_BES_df) label = "" marker = (:rtriangle, :dodgerblue)
    @plt_data! Data(data_πK10_BaBar_df) label = "" marker = (:ltriangle, :dodgerblue)
    
    @plt_data! Data(data_πK11_Estabrooks_df)[1:23] label = L"\delta_{1/2}^1" marker = (:rect, :darkorange)
    @plt_data! Data(data_πK11_Aston_df)[1:21] label = "" marker = (:diamond, :darkorange)
    @plt_data! Data(data_πK11_Mercer_df)[1:14] label = "" marker = (:utriangle, :darkorange)
 
    @plt_data! Data(data_πK30_Estabrooks_df)[1:10] label = L"\delta_{3/2}^0" marker = (:rect, :green3)
    @plt_data! Data(data_πK30_Mercer_df)[1:14] label="" marker = (:utriangle, :green3)
    @plt_data! Data(data_πK30_Lingin_df) label="" marker=(:circle, :darkgreen) xlab=L"\sqrt{s}"*" [MeV]" ylab = L"\delta_{J}^I (K\pi\to K \pi)"
    hline!([0], color = :black, label=:none)   
    
    wv2 = 985:1.0:1200
    p3 = plot(wv2, δππkk00.(wv2, _lec), label = "", xlim = (wv2[1], wv2[end]), ylim=(0,260) )
    @plt_data! Data(data_ππkk00_Cohen_df) label = "" marker = :darkorange
    @plt_data! Data(data_ππkk00_Martin_df)[1:9] label="" marker=(:rect,:darkorange) xlab=L"\sqrt{s}"*" [MeV]" ylab = L"\delta_{0}^0 (\pi\pi\to K \bar K)"
       
    p4 = plot(wv2, η00.(wv2, _lec), legend = :none, xlim = (wv2[1], wv2[end]), ylim=(0, 1.6))
    @plt_data! data_ππ00_eta marker = :darkorange xlab=L"\sqrt{s}"*" [MeV]" ylab = L"\eta_{0}^0" # label = L"\eta_{0}^0"
    
    wv3 = (mpi+meta+0.000001):1.0:1075
    p5 = plot(wv3, σπη.(wv3, Ref(paras)), label = "", xlim = (wv3[1], wv3[end]) )
    @plt_data! data_πη[1:16] label = "" 
    plot!(wv3, data_πη_WA76_bg.(wv3), line=(:dashdot, :gray), label="background" )
    plot!(xlab=L"\sqrt{s}"*" [MeV]", ylab = L"d\sigma_{\pi\eta}/d\sqrt{s}"*" [events/25 MeV]", ylim=(0, 320))
    
    plot(p1, p2, p3, p4, p5, size=(1200,550))
end

In [ ]:
@time plt_compare([lecr0..., 1e-4]; jump = 900)

In [ ]:
function plt_unitarity_test(tm, ch1, ch2; lec=lecr0, kwds...) 
#     ch1 = ππ; ch2 = KK;
    wv = (ch1.m1+ch1.m2):1:1100 #(ch2.m1+ch2.m2)
    tij(w) = tm(w, lec)  
    _im1(w) = imag(tij(w)[1,1])
    _re1(w) = real(tij(w)[1,1])
    _abs2(w) = abs2(tij(w)[1,1])
    _rho1(w) = qon(w^2, ch1.m1, ch1.m2)/(8π*w); _rho2(w) = qon(w^2, ch2.m1, ch2.m2)/(8π*w)
    _rhs(w) = (w <= ch2.m1+ch2.m2 ? _abs2(w) * _rho1(w) : 
        _abs2(w) * _rho1(w) + abs2(tij(w)[1,2])*_rho2(w))
    p1 = plot(wv, _im1.(wv); label = "Im "*L"T", kwds... )
    plot!(wv, _rhs.(wv); label = L"T\Sigma T^\dag", line = :dash )
    plot!(wv, _re1.(wv); label = "Re "*L"T", line = :dot )
end

# plotly()
function plt_unitarity(lec)
    p1 = plt_unitarity_test(Stiam_00, ππ, KK, lec=lec, ylab = L"T_0^0[\pi\pi\to \pi\pi]")
    p2 = plt_unitarity_test(Stiam_11, ππ, KK, lec=lec, xlab = L"\sqrt{s}"*" [MeV]", ylab = L"T_1^1[\pi\pi\to \pi\pi]")
    p3 = plt_unitarity_test(Stiam_10, πη, KK, lec=lec, xlab = L"\sqrt{s}"*" [MeV]", ylab = L"T_1^0[\pi\eta\to \pi\eta]")
    p4 = plt_unitarity_test(Stiam_strange_10, Kπ, Kη, lec=lec, ylab = L"T_0^{1/2}[K\pi\to K\pi]")
    p5 = plt_unitarity_test(Stiam_strange_11, Kπ, Kη, lec=lec, xlab = L"\sqrt{s}"*" [MeV]", ylab = L"T_1^{1/2}[K\pi\to K\pi]")
    plot(p1, p2, p3, p4, p5, layout = 5, size=(1200,500))
end

In [ ]:
include("src/tmatrix.jl")

In [ ]:
pyplot()# plotly() #to make numerical comparison visible, use plotly()
plt_unitarity(lecr0)

In [ ]:
include("src/unitarity_modification.jl");

In [ ]:
plt_unitarity(lecr0)
# savefig("unitarity_mod.pdf")

In [ ]:
# pyplot()
plt_unitarity(lec_chpt)

In [ ]:
function plt_iam(tiam, lec)
    wv = 200:2:1200
    n = length(tiam(300, lec)) 
    p = Vector{Any}(undef, n)
    for i = 1:n
        stm11(w) = tiam(w, lec)[i]
        p[i] = plot( wv, (@. abs(stm11(wv)) ), label="|T($i)|" )
        plot!( wv, (@. real(stm11(wv)) ), line = :dash, label="Re T($i)" )
        plot!( wv, (@. imag(stm11(wv)) ), line = :dot, label="Im T($i)" )
    end
    plot(p..., layout = n, size=(700,500)) #, legend=:outertopright)
end

In [ ]:
# NLO amplitudes
@time plt_iam((w, lec)->St4_00!(cmatrix33, w, lec), lecr0)

In [ ]:
@time plt_iam(Stiam_00, lecr0)

In [ ]:
@time plt_iam(Stiam_10, lecr0)

In [ ]:
@time plt_iam(Stiam_11, lecr0)

In [ ]:
let wv = 600:0.5:1200, amp
    amp(w) = tiam_01(w, lecr0)
    plot( wv, (@. abs(amp(wv)) ), label=L"|T_{01}|", size=(500,350), xlab=L"\sqrt{s}"*" [MeV]", ylab=L"T_{01}")
    plot!( wv, (@. real(amp(wv)) ), line=:dash, label="Re "*L"T_{01}")
    plot!( wv, (@. imag(amp(wv)) ), line=:dot, label="Im "*L"T_{01}")
    vline!([sqrt(4mk^2-4mpi^2)], line=(:dashdot,:gray), label=L"2\sqrt{M_K^2-M_\pi^2}" )
    plot!(title=L"K\bar K\to K\bar K (I=0, J=1)")
    
end

# Fits

Following [Kaminski et al., Phys.Lett.B 551 (2003) 241-248](https://inspirehep.net/literature/600560), the chi-square for fitting to the phase shifts can be defined as
\begin{equation}
  \chi^2 = \sum_{i = 1}^N \left[ \frac{\sin\left(\delta_\ell^I(s_i) - \varphi_\ell^I(s_i) \right) }{\Delta\varphi_\ell^I(s_i)} \right]^2,
\end{equation}
where $\varphi_\ell^I(s_i)$ and $\Delta\varphi_\ell^I(s_i)$ are the phase shift data and the errors.
This should be understood as the phases are expressed in units of radians.

In [ ]:
# as defined above
function chisq_ps(dist::Function, data::Data, par; fitrange = ())
    fitrange = (isempty(fitrange) ? (1:data.ndata) : fitrange)
    res = 0.0
    @simd for i = fitrange[1]:fitrange[end]
        @inbounds res += ( sind(data.y[i]- dist(data.x[i], par))/(data.err[i]*π/180) )^2
    end
    return res
end

# to ensure continuity of phase shifts in resonant partial waves; unneessary in the case of using sin
function chisq_continuity(dist::Function, data::Data, par; fitrange = ())
    fitrange = (isempty(fitrange) && 1:data.ndata)
    res = 0.0
     _ps0 = 0.0; _jump = -20.0
    @simd for i = fitrange
        _ps = dist(data.x[i], par)
        if _ps - _ps0 < _jump  
            _ps += 180.0
        end
        _ps0 = _ps
        @inbounds res += ( sind(data.y[i] - _ps)/(data.err[i]*π/180) )^2
    end
    return res
end

# here define chisq to let 2L6+L8 be treated a linear combination
for str in ["chisq", "chisq_continuity", "chisq_ps"]
    _chsq68 = Symbol("$(str)_68")
    _chsq = Symbol("$(str)")
    _expr = quote
        function ($_chsq68)(f, data, pars) 
            _pars = zeros(length(pars))
            for i = 1:7 
                @views _pars[i] = pars[i]
            end
            @views _pars[8] = pars[8] - 2pars[6]
            return ($_chsq)(f, data, _pars)
        end
    end
    eval(_expr)
end


In [ ]:
const lec_for_best = (0.3461, 0.9102, -2.6452, -0.5261, 1.03, 0.07, -0.62, 1.11) .*1e-3;

In [ ]:
name = [:L1, :L2, :L3, :L4, :L5, :L6, :L7, :L8, :c];

In [ ]:
@time chisq_continuity(δhalf0, data_πK10, lecr0), chisq_ps(δhalf0, data_πK10, lecr0)

In [ ]:
function plt_paras(df::DataFrame, col1, col2)
    scatter(df[!, col1], df[!, col2], xlab = col1, ylab=col2, label = :none)
end

# show scatter plots of all parameter combinations
function plt_paras(df::DataFrame)
    paras = (:L1, :L2, :L3, :L4, :L5, :L7, :L8)
    plt = Array{Any, 2}(undef, 7,7)
    for i in eachindex(paras)
        for j in eachindex(paras) 
            plt[i,j] = plt_paras(df, paras[i], paras[j])
        end
    end
    plot(plt..., size = (1350, 900))
end

In [ ]:
# for plotting error bands
function dist_minmax(df::DataFrame, fitfun, w)
        m = size(df)[1]
        _tempvec = zeros(m)
        @inbounds @simd for i = 1:m      
            _tempvec[i] = fitfun(w, Array(df[i, 2:end]))
        end
        return [extrema(_tempvec)...]
end

function plt_band(df::DataFrame, ch::TwoBodyChannel, fitfun; color= :gray, kws...) 
    dist_extrema(w) = dist_minmax(df, fitfun, w)  
    n = 100; 
    wv = LinRange(ch.m1+ch.m2-2, 1200, n)
    paras = Array(df[1, 2:end])
    _yv = permutedims(reduce(hcat, dist_extrema.(wv)))  
    plot( wv, [_yv[:,1]]; color= color, linealpha=0.5, fill = ([_yv[:,2]], color), fillalpha = 0.5, label="", kws... )
end
function plt_band!(df::DataFrame, ch::TwoBodyChannel, fitfun; color= :gray, kws...)
    dist_extrema(w) = dist_minmax(df, fitfun, w)  
    n = 100; 
    wv = LinRange(ch.m1+ch.m2-2, 1200, n)
    best = zeros(Float64, n); 
    paras = Array(df[1, 2:end])
    _yv = permutedims(reduce(hcat, dist_extrema.(wv)))  
    plot!( wv, [_yv[:,1]]; color= color,linealpha=0.5, label="", fill = ([_yv[:,2]], color), fillalpha = 0.5, kws...)
end

## using the errors from the data

In [ ]:
χsq_00(pars) = chisq_ps(δ00_0, data_ππ00, @views pars[1:8] )
χsq_11(pars) = chisq_ps(δ11, data_ππ11, @views pars[1:8] ) 
χsq_20(pars) = chisq_ps(δ20, data_ππ20, @views pars[1:8] )
χsq_half0(pars) = chisq_ps(δhalf0, data_πK10, @views pars[1:8])
χsq_half1(pars) = chisq_ps(δhalf1, data_πK11, @views pars[1:8])
χsq_3half0(pars) = chisq_ps(δ3half0, data_πK30, @views pars[1:8])
χsq_η00(pars) = chisq(η00, data_ππ00_eta, @views pars[1:8] )
# for this one, needs to check whether the minimum is such that the phase is too small by 180 degrees
χsq_ππkk00(pars) = chisq_ps(δππkk00, data_ππKK00, @views pars[1:8] );

χsq_πη(pars) = chisq(σπη, data_πη[1:16], pars );

In [ ]:
@time χsq_πη(paras0)

In [ ]:
data_ππ00.ndata, data_ππ11.ndata, data_ππ20.ndata, data_πK10.ndata, data_πK11.ndata, data_πK30.ndata,
data_ππKK00.ndata, data_ππ00_eta.ndata, data_πη.ndata

In [ ]:
macro show_individual(lec)
    summing = 0.0
    _list = ["00", "11", "20", "half0", "half1", "3half0", "ππkk00", "η00", "πη"]
    dofs = (31, 31, 23, 103, 58, 36, 17, 30, 16)
    for i in eachindex(_list)
        func = Symbol("χsq_$(_list[i])")
        res = :(($func)($lec))
        @eval println("χ^2/dof in ", $(_list[i]), ":  ", $res/($(dofs[i])-8) )
        summing += eval(res)
    end
    @eval println("Total χ^2/dof:   ", $summing/(sum($dofs) -8) )
end

In [ ]:
χsq(pars) = (χsq_00(pars)  + χsq_11(pars) + χsq_20(pars) + χsq_ππkk00(pars) + χsq_η00(pars)
             + χsq_half0(pars) + χsq_half1(pars) + χsq_3half0(pars) + χsq_πη(pars) 
    )
fit0 = Minuit(χsq, paras0, error=1e-6*ones(9), name = name,
     fix_L1=false, fix_L3=false, fix_L4=false, fix_L5=false, fix_L6=true, fix_L7=false, fix_L8=false
)
fit0.strategy = 2;

In [ ]:
@show_individual  paras0

In [ ]:
@time migrad(fit0)

In [ ]:
@show_individual  args(fit0)

In [ ]:
@time hesse(fit0)
@time hesse(fit0)

In [ ]:
@time minos(fit0)

In [ ]:
pyplot()
plt_unitarity(args(fit0))

In [ ]:
# pyplot()
@time plt_compare(args(fit0), jump = 950)

In [ ]:
view(args(fit0),1:8)

### with increased errors following the paper by Gomez Nicola and Pelaez

In [ ]:
const add_err = 0.05; 
const lec_best0 = [0.522, 1.081, -2.7, -0.279, 1.87, 0.07, -1.29, 1.70] .*1e-3;

In [ ]:
# increase the data error by  `factor` times of the data value
# weight(dat::Data, factor) = Data(dat.x, dat.y, dat.err .* factor)
weight(dat::Data, factor) = Data(dat.x, dat.y, (@. dat.err + factor * abs(dat.y)) );

In [ ]:
χsq_pc_00(pars) = chisq_ps(δ00_0, weight(data_ππ00, add_err), @views pars[1:8])
χsq_pc_11(pars) = chisq_ps(δ11, weight(data_ππ11, add_err), @views pars[1:8]) 
χsq_pc_20(pars) = chisq_ps(δ20, weight(data_ππ20, add_err), @views pars[1:8])
χsq_pc_η00(pars) = chisq(η00, weight(data_ππ00_eta, add_err), @views pars[1:8]);
χsq_pc_ππkk00(pars) = chisq_ps(δππkk00, weight(data_ππKK00, add_err), @views pars[1:8])

χsq_pc_half0(pars) = chisq_ps(δhalf0, weight(data_πK10, add_err), @views pars[1:8])
χsq_pc_half1(pars) = chisq_ps(δhalf1, weight(data_πK11, add_err), @views pars[1:8])
χsq_pc_3half0(pars) = chisq_ps(δ3half0, weight(data_πK30, add_err), @views pars[1:8]);

χsq_pc_πη(pars) = chisq(σπη, weight(data_πη[1:16], add_err), pars);

In [ ]:
macro show_individual_pc(lec)
    summing = 0.0
    _list = ["00", "11", "20", "half0", "half1", "3half0", "ππkk00", "η00", "πη"]
    dofs = (31, 31, 23, 103, 58, 36, 17, 30, 16)
    for i in eachindex(_list)
        func = Symbol("χsq_pc_$(_list[i])")
        res = :(($func)($lec))
        @eval println("χ^2/dof in ", $(_list[i]), ":  ", $res/($(dofs[i])-8) )
        summing += eval(res)
    end
    @eval println("Total χ^2/dof:   ", $summing/(sum($dofs) -8) )
end

In [ ]:
@show_individual_pc paras0

In [ ]:
@show_individual_pc args(fit0)

In [ ]:
χsq_pc(pars) = (χsq_pc_00(pars)  + χsq_pc_11(pars) + χsq_pc_20(pars) + χsq_pc_ππkk00(pars) + χsq_pc_η00(pars)
             + χsq_pc_half0(pars) + χsq_pc_half1(pars) + χsq_pc_3half0(pars) + χsq_pc_πη(pars)
    )
fit_pc = Minuit(χsq_pc, paras0, error=1e-6*ones(9), name = name,
     fix_L1=false, fix_L3=false, fix_L4=false, fix_L5=false, fix_L6=true, fix_L7=false, fix_L8=false
)
fit_pc.strategy = 2;

In [ ]:
@time migrad(fit_pc)

In [ ]:
lec_best_plus5pc = [0.546, 1.07, -2.756, -0.202, 1.1, 0.07, -0.52, 0.68, 0.109] .*1e-3;

In [ ]:
@show_individual_pc args(fit_pc)

In [ ]:
@time hesse(fit_pc)
@time hesse(fit_pc)

In [ ]:
@time minos(fit_pc)

In [ ]:
@time migrad(fit_pc)

In [ ]:
matrix(fit_pc; correlation=true)

In [ ]:
@time let p1, p2, lec = view(args(fit0), 1:8), lec2 = view(args(fit_pc), 1:8)
    p1 = plot(mpi+meta:1:1200, x->δ10_πη_0(x,lecr0), ylab = L"\delta_0^1(\pi\eta)", label="original")
    plot!(mpi+meta:1:1200, x->δ10_πη_0(x, lec), line=:dash, label="Fit 1" )
    plot!(mpi+meta:1:1200, x->δ10_πη_0(x, lec2), line=:dot, label="Fit 2" )
    p2 = plot(2mk:1:1200, x->δ10_kk(x,lecr0), ylab = L"\delta_0^1(K\bar K)", label="original")
    plot!(2mk:1:1200, x->δ10_kk(x,lec), line=:dash, label="Fit 1")
    plot!(2mk:1:1200, x->δ10_kk(x,lec2), line=:dot, label="Fit 2")
    p3 = plot(2mk:1:1200, x->δ10_πη_0(x,lecr0)+δ10_kk(x,lecr0), ylab = L"\delta_0^1(\pi\eta+K\bar K)", label="original")
    plot!(2mk:1:1200, x->δ10_πη_0(x,lec)+δ10_kk(x,lec), line=:dash, label="Fit 1")
    plot!(2mk:1:1200, x->δ10_πη_0(x,lec2)+δ10_kk(x,lec2), line=:dot, label="Fit 2")
    plot(p1, p2, p3, layout=(1,3), size = (1000,300), xlab=L"\sqrt{s}"*" [MeV]", ylim=(0,58))
end 

The above phase shifts are different from those in the following two papers:

* [1507.04526](https://arxiv.org/abs/1507.04526): *Form factors of the isovector scalar current and the ηπ scattering phase shifts*
by M. Albaladejo, B. Moussallam
* [2002.04441](https://arxiv.org/abs/2002.04441): *The πη interaction and $a_0$ resonances in photon-photon scattering* by
Junxu Lu, B. Moussallam
<img src="./datajl/pieta_150704526.png" style="height: 250px;"/>  <img src="./datajl/pieta_200204441.png" style="height: 250px;"/> 

One reason for the difference would be that these analyses have higher $a_0$ states built in, while the UCHPT does not. 

In [ ]:
plt_unitarity(args(fit_pc) )

In [ ]:
@time plt_compare(args(fit_pc), jump = 950)

## Parameter error bands from MC-Δχ² sampling

Sample the joint 1σ region around the best fit with JuMinuit's native
[`contour_df_samples`](https://fkguo.github.io/JuMinuit.jl/dev) (the MC-Δχ² method — each accepted set carries its *true* Δχ²), then take the
min/max envelope of the predicted curves over the physical sets as the error band.

In [ ]:
# JuMinuit-native error-band sampling (replaces IMinuit.jl's contour-tracing
# `contour_df`). `contour_df_samples` Monte-Carlo–samples the joint 1σ region
# (cl=1, ndof defaults to the number of free parameters) and keeps each set's
# TRUE Δχ². See `?get_contours_samples` and docs/src/error_analysis.md.
@time samples_pc = contour_df_samples(fit_pc; χsq = χsq_pc, cl = 1, nsamples = 2000, seed = 1234)

# Reshape for the band plotters (`plt_band` reads `df[i, 2:end]`): column 1 =
# absolute χ², columns 2:end = the full 9-parameter vector in order, with the
# fixed L6 re-inserted at its best-fit value.
parasdf_plus5pc = DataFrame(χ² = samples_pc.delta_chisq .+ fit_pc.fval)
for (i, nm) in enumerate(name)
    parasdf_plus5pc[!, nm] = is_fixed(fit_pc.params.pars[i]) ?
        fill(Float64(fit_pc.values[i]), nrow(samples_pc)) : samples_pc[!, nm]
end
parasdf_plus5pc

In [ ]:
CSV.write("paras_plus5pc_mncount.csv", parasdf_plus5pc)

In [ ]:
parasdf_plus5pc = DataFrame(CSV.File("paras_plus5pc_mncount.csv"));

In [ ]:
# Pick out parameter sets whose χ² dips BELOW the best fit — a hallmark of an
# unphysical singularity that artificially lowers χ² at some point (an abrupt
# change of χ² in the 00 channel). The rest are the physical 1σ band.
# (`fit_pc.fval` is the best-fit χ²; the old notebook hard-coded it as 406.4.)
suspected = parasdf_plus5pc[parasdf_plus5pc.χ² .< fit_pc.fval, :];
physical_candidates = parasdf_plus5pc[parasdf_plus5pc.χ² .>= fit_pc.fval, :];

In [ ]:
pyplot()
@time plt_paras(physical_candidates .* 1e3)

In [ ]:
# it turns out that all the parameter sets in physical_candidates do not have unphysical singularities
pyplot()
let p1, p2, p3
    p1 = plt_band(physical_candidates, ππ, (x, p)->δ00(x,p, jump=930))
    plt_band!(physical_candidates, ππ, δ11)
    plt_band!(physical_candidates, ππ, δ20)
    p2 = plt_band(physical_candidates, Kπ, δhalf0)
    plt_band!(physical_candidates, Kπ, δhalf1)
    plt_band!(physical_candidates, Kπ, δ3half0)
    p3 = plt_band(physical_candidates, KK, η00) #πη, σπη)

    plot(p1, p2, p3, layout= (1, 3), size=(1200,300))
end

In [ ]:
CSV.write("paras_plus5pc_physical.csv", physical_candidates)

In [ ]:
function plt_compare2(lec, lec_pc, bands::DataFrame=physical_candidates; jump = 930)
    _lec = Ref(lec); _lec0 = Ref(paras0); _paras = bands; _lecpc = Ref(lec_pc);
    wv = 270:1.0:1200
    p1 = plot(wv, δ00.(wv, _lec; jump = jump), line=:dot, label="", xlim = (wv[1], wv[end]))
    plt_band!(_paras, ππ, (x, p)->δ00(x,p, jump=920), color=:dodgerblue)
    plot!(wv, δ00.(wv, _lecpc; jump = jump), color=:dodgerblue, label="" )
    plot!(wv, δ11.(wv, _lec), line=:dot, color=:darkorange, label="" )
    plot!(wv, δ11.(wv, _lecpc), color=:darkorange, label="" )
    plt_band!(_paras, ππ, δ11, color=:darkorange)
    plot!(wv, δ20.(wv, _lec), label="", color=:green3, line=:dot )
    plot!(wv, δ20.(wv, _lecpc), color=:green3, label="" )
    plt_band!(_paras, ππ, δ20, color=:green3)
    plot!(wv, δ00.(wv, _lec0; jump = jump), label="", line=:dash, color=:dodgerblue  )
    plot!(wv, δ11.(wv, _lec0), label="", line=:dash, color=:darkorange )
    plot!(wv, δ20.(wv, _lec0), label="", line=:dash, color=:green3)
    @plt_data! data_ππ00 label = L"\delta_{0}^0" marker = (:circle, :dodgerblue)
    @plt_data! data_ππ11 label = L"\delta_{1}^1" marker = (:circle, :darkorange)
    @plt_data! data_ππ20 label = L"\delta_{0}^2" marker = (:circle, :green3) xlab=L"\sqrt{s}"*" [MeV]" ylab = L"\delta_{J}^I (\pi\pi\to \pi\pi)"
    hline!([0], color = :black, label=:none)    
    
    xv = 630:1.0:1200
    p2 = plot(xv, δhalf0.(xv, _lec), line=:dot, label="", xlim = (xv[1], xv[end]))
    plot!(xv, δhalf0.(xv, _lecpc), color=:dodgerblue, label="" )
    plt_band!(_paras, Kπ, δhalf0, color=:dodgerblue)
    plot!(xv, δhalf1.(xv, _lec), line=:dot, color=:darkorange, label="" )
    plot!(xv, δhalf1.(xv, _lecpc), color=:darkorange, label="" )
    plt_band!(_paras, Kπ, δhalf1, color=:darkorange)
    plot!(xv, δ3half0.(xv, _lec), line=:dot, color=:green3, label="" )
    plot!(xv, δ3half0.(xv, _lecpc), color=:green3, label="" )
    plt_band!(_paras, Kπ, δ3half0, color=:green3)
    plot!(wv, δhalf0.(wv, _lec0), label="", line=:dash, color=:dodgerblue  )
    plot!(wv, δhalf1.(wv, _lec0), label="", line=:dash, color=:darkorange )
    plot!(wv, δ3half0.(wv, _lec0), label="", line=:dash, color=:green3)
    plot!(wv, δhalf0.(wv, _lec0), label="", line=:dash, color=:dodgerblue  )
    plot!(wv, δhalf1.(wv, _lec0), label="", line=:dash, color=:darkorange )
    plot!(wv, δ3half0.(wv, _lec0), label="", line=:dash, color=:green3)
    @plt_data! Data(data_πK10_Estabrooks_df)[1:19] label = L"\delta^{1/2}_0" marker = (:rect, :dodgerblue)
    @plt_data! Data(data_πK10_Aston_df)[1:21] label = "" marker = (:diamond, :dodgerblue)
    @plt_data! Data(data_πK10_Mercer_df)[1:14] label = "" marker = (:utriangle, :dodgerblue)
    @plt_data! Data(data_πK10_Bingham_df)[1:19] label = "" marker = (:dtriangle, :dodgerblue)
    @plt_data! Data(data_πK10_Baker_df) label = "" marker = (:pentagon, :dodgerblue)
    @plt_data! Data(data_πK10_BES_df) label = "" marker = (:rtriangle, :dodgerblue)
    @plt_data! Data(data_πK10_BaBar_df) label = "" marker = (:ltriangle, :dodgerblue)
    
    @plt_data! Data(data_πK11_Estabrooks_df)[1:23] label = L"\delta^{1/2}_1" marker = (:rect, :darkorange)
    @plt_data! Data(data_πK11_Aston_df)[1:21] label = "" marker = (:diamond, :darkorange)
    @plt_data! Data(data_πK11_Mercer_df)[1:14] label = "" marker = (:utriangle, :darkorange)
 
    @plt_data! Data(data_πK30_Estabrooks_df)[1:10] label = L"\delta^{3/2}_0" marker = (:rect, :green3)
    @plt_data! Data(data_πK30_Mercer_df)[1:14] label="" marker = (:utriangle, :green3)
    @plt_data! Data(data_πK30_Lingin_df) label="" marker=(:circle, :green3) xlab=L"\sqrt{s}"*" [MeV]" ylab = L"\delta_{J}^I (K\pi\to K \pi)"
    hline!([0], color = :black, label=:none)   
    
    wv2 = 985:1.0:1200
    p3 = plot(wv2, δππkk00.(wv2, _lec), line=:dot, label = "", xlim = (wv2[1], wv2[end]), ylim=(0,260) )
    plot!(wv2, δππkk00.(wv2, _lecpc), label="", color=:dodgerblue)
    plt_band!(_paras, KK, δππkk00, color=:dodgerblue)
    plot!(wv2, δππkk00.(wv2, _lec0), label="", line=:dash, color=:dodgerblue)
    @plt_data! Data(data_ππkk00_Cohen_df) label = "" marker = :dodgerblue
    @plt_data! Data(data_ππkk00_Martin_df)[1:9] label="" marker=(:rect,:dodgerblue) xlab=L"\sqrt{s}"*" [MeV]" ylab = L"\delta_{0}^0 (\pi\pi\to K \bar K)"
       
    p4 = plot(wv2, η00.(wv2, _lec), line=:dot, legend = :none, xlim = (wv2[1], wv2[end]), ylim=(0, 1.6))
    plot!(wv2, η00.(wv2, _lecpc), color=:dodgerblue)
    plt_band!(_paras, KK, η00, color=:dodgerblue)
    plot!(wv2, η00.(wv2, _lec0), label="", line=:dash, color=:dodgerblue)
    @plt_data! data_ππ00_eta marker = :dodgerblue xlab=L"\sqrt{s}"*" [MeV]" ylab = L"\eta_{0}^0" # label = L"\eta_{0}^0"
    
    wv3 = (mpi+meta+0.000001):1.0:1075
    p5 = plot(wv3, σπη.(wv3, _lec), line=:dot, label = "", xlim = (wv3[1], wv3[end]) )
    plot!(wv3, σπη.(wv3, _lecpc), label="",  color=:dodgerblue)
    plt_band!(_paras, πη, σπη, color=:dodgerblue)
    plot!(wv3, σπη.(wv3, _lec0), label="", line=:dash, color=:dodgerblue)
    @plt_data! data_πη[1:16] label = "" marker = :dodgerblue
    plot!(wv3, data_πη_WA76_bg.(wv3), line=(:dashdot, :gray), label="background" )
    plot!(xlab=L"\sqrt{s}"*" [MeV]", ylab = L"d\sigma_{\pi\eta}/d\sqrt{s}"*" [events/25 MeV]", ylim=(0, 320))
    
    plot(p1, p2, p3, p4, p5, plot(), layout =@layout([a b; c d; e _]), 
        size=(700,700), legendfontsize = 8, guidefont = 9)
end

In [ ]:
@time plt_compare2(args(fit0), args(fit_pc), physical_candidates)
savefig("iam_bestfit.pdf") #, bbox_inches="tight", pad_inches=0

In [ ]:
# expars = map(extrema, eachcol(physical_candidates))
let _pars = map(extrema, eachcol(physical_candidates))
    [_pars[i+1] .- args(fit_pc)[i] for i in 1:9 ]
end

### with increased errors except for those from DR analysis

In [ ]:
add_moreerr = 0.1;

In [ ]:
χsq_pc_00(pars) = chisq_ps(δ00_0, data_ππ00, @views pars[1:8])
χsq_pc_11(pars) = chisq_ps(δ11, data_ππ11, @views pars[1:8]) 
χsq_pc_20(pars) = chisq_ps(δ20, data_ππ20, @views pars[1:8])
χsq_pc_η00(pars) = chisq(η00, weight(data_ππ00_eta, add_moreerr), @views pars[1:8]);
χsq_pc_ππkk00(pars) = chisq_ps(δππkk00, weight(data_ππKK00, add_moreerr), @views pars[1:8])

χsq_pc_half0(pars) = chisq_ps(δhalf0, weight(data_πK10, add_moreerr), @views pars[1:8])
χsq_pc_half1(pars) = chisq_ps(δhalf1, weight(data_πK11, add_moreerr), @views pars[1:8])
χsq_pc_3half0(pars) = chisq_ps(δ3half0, weight(data_πK30, add_moreerr), @views pars[1:8]);

χsq_pc_πη(pars) = chisq(σπη, weight(data_πη[1:16], add_err), pars);

In [ ]:
@show_individual_pc paras0

In [ ]:
@show_individual_pc lec_best_plus5pc

In [ ]:
χsq_pc(pars) = (χsq_pc_00(pars)  + χsq_pc_11(pars) + χsq_pc_20(pars) + χsq_pc_ππkk00(pars) + χsq_pc_η00(pars)
               + χsq_pc_πη(pars)
             + χsq_pc_half0(pars) + χsq_pc_half1(pars) + χsq_pc_3half0(pars)
    )
fit_pc = Minuit(χsq_pc, paras0, error=1e-6*ones(9), name = name,
     fix_L1=false, fix_L3=false, fix_L4=false, fix_L5=false, fix_L6=true, fix_L7=false, fix_L8=false
)
fit_pc.strategy = 2;

In [ ]:
@time migrad(fit_pc)

In [ ]:
@show_individual_pc args(fit_pc)

In [ ]:
plt_compare(args(fit_pc), jump = 850)